# Offline evaluation on `shop.history_sessions`

Before putting a new recommender live, estimate offline whether it beats the baseline.

The log was produced by the **old system** (best sellers plus a bit of randomness), so it only tells you what happened with the products *it* showed. Two estimates:

1. **Hit rate**: of the impressions where the shopper clicked, how often is the clicked product in your top 10? Easy, but biased towards popular products (the old system showed them most).
2. **Inverse propensity scoring (IPS)** of the click rate in **slot 1**: keep only the impressions where your first product equals the one the old system put in slot 1, and weight each by `1 / propensity`. This corrects the popularity bias of the log for that slot. SNIPS (self-normalised IPS) has lower variance.

We hold out the **last 7 days** of the history: the policies are built only from data before the cut and evaluated after it.

In [1]:
import os
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import requests

for line in Path("../.env").read_text().splitlines() if Path("../.env").exists() else []:
    key, _, value = line.partition("=")
    os.environ.setdefault(key.strip(), value.strip())
API = os.environ.get("RAWTREE_API_URL", "https://api.rawtree.com")
http = requests.Session()
http.headers["Authorization"] = f"Bearer {os.environ['RAWTREE_API_KEY']}"

def sql(q):
    r = http.post(f"{API}/v1/query", json={"sql": q}, timeout=600)
    r.raise_for_status()
    return pd.DataFrame(r.json()["data"])

## Data

In [2]:
from pathlib import Path

last_day = sql("SELECT toDate(max(ts::DateTime64(3))) AS d FROM shop.history_sessions").d[0]
CUT = str((pd.Timestamp(last_day) - pd.Timedelta(days=7)).date())   # 'YYYY-MM-DD'
print("train < ", CUT, "<= test")

TEST_CACHE = Path("test_cut.pkl")
if TEST_CACHE.exists():
    test = pd.read_pickle(TEST_CACHE)
else:
    test = sql(f"""
        SELECT request_id, ts, user_id, context, recent_views, shown, propensity, clicked, purchased
        FROM shop.history_sessions WHERE ts >= '{CUT}' ORDER BY rand() LIMIT 100000""")
    test["clicked"] = test.clicked.replace("", None)
    test.to_pickle(TEST_CACHE)

print(len(test), "test impressions, observed CTR", round(test.clicked.notna().mean(), 4))

train <  2026-11-07 <= test
100000 test impressions, observed CTR 0.0373


In [3]:
# Co-views computed inside RawTree: for every product, the 20 products most viewed in the same sessions
coviews = sql(f"""
    SELECT a.product_id AS p, b.product_id AS q, count() AS n
    FROM (SELECT session_id::String AS session_id, product_id::String AS product_id FROM shop.history_events WHERE event_type = 'view' AND ts < '{CUT}') AS a
    INNER JOIN (SELECT session_id::String AS session_id, product_id::String AS product_id FROM shop.history_events WHERE event_type = 'view' AND ts < '{CUT}') AS b
      ON a.session_id = b.session_id
    WHERE a.product_id != b.product_id
    GROUP BY p, q ORDER BY n DESC LIMIT 20 BY p""")
covisit = coviews.groupby("p").q.apply(list).to_dict()
print(len(covisit), "products with co-views")

5000 products with co-views


In [4]:
import os, sys
os.environ["MODEL_SUFFIX"] = "_cut"
sys.path.insert(0, ".")
for m in ("recommender7", "features7"):
    sys.modules.pop(m, None)
import recommender7 as ours
from features7 import add_hourly, make_ctx, rank_request, top10, viewed_from_requests

# Los 200 más vendidos ANTES del corte (el baseline real, sin ver el futuro)
best = sql(f"""
    SELECT product_id::String AS product_id, count() AS n
    FROM shop.history_events
    WHERE event_type = 'purchase' AND ts::DateTime64(3) < toDateTime('{CUT}')
    GROUP BY product_id ORDER BY n DESC LIMIT 200""").product_id.tolist()

# Vistas por subcategoría y hora del periodo de test: la tendencia se calcula punto-en-el-tiempo
hv_rows = sql(f"""
    SELECT c.subcategory::String AS sub, toUnixTimestamp(toStartOfHour(e.ts::DateTime64(3))) AS h, count() AS v
    FROM shop.history_events AS e
    INNER JOIN shop.catalog AS c ON c.product_id::String = e.product_id::String
    WHERE e.event_type = 'view' AND e.ts::DateTime64(3) >= toDateTime('{CUT}') - INTERVAL 26 HOUR
    GROUP BY sub, h""")
add_hourly(ours.STATE, hv_rows.to_dict("records"))
print(len(hv_rows), "celdas horarias añadidas; modelo:", ours.MODELS["version"], "| exponente compra:", ours.MODELS["buy_exp"])

17596 celdas horarias añadidas; modelo: v7-ctx-trend-buy | exponente compra: 1.0


## Policies

A policy maps one request (a row) to an ordered list of 10 product IDs.

In [5]:
test = test.sample(25000, random_state=0)      # 25k filas bastan y evitan esperar 15 min

def baseline(row):
    return best[:10]

def coview(row):
    score = Counter()
    for k, v in enumerate(reversed(row.recent_views or [])):
        for rank, q in enumerate(covisit.get(v["product_id"], [])):
            score[q] += 0.7 ** k / (rank + 1)
    items = [q for q, _ in score.most_common(10)]
    return (items + [p for p in best if p not in items])[:10]

def ours_policy(row):
    viewed = viewed_from_requests(row.recent_views)
    ctx = make_ctx(row.context, row.ts, state=ours.STATE)
    return top10(rank_request(ours.STATE, ours.MODELS, ctx, viewed, ours.ACTIVE), best)

_memo = {}                                      # cada política se calcula una sola vez por fila
def memo(name, f):
    def g(row):
        k = (name, row.Index)
        if k not in _memo:
            _memo[k] = f(row)
        return _memo[k]
    return g

POLICIES = {k: memo(k, f) for k, f in {"baseline": baseline, "coview": coview, "ours": ours_policy}.items()}


## Evaluation

In [6]:
def evaluate(policy, df):
    recs = [policy(r) for r in df.itertuples()]
    clicked = df.clicked.to_numpy()
    has_click = df.clicked.notna().to_numpy()
    hit = np.array([c in r for c, r in zip(clicked, recs)])[has_click].mean()
    first_shown = df.shown.str[0].to_numpy()
    match = np.array([r[0] == s for r, s in zip(recs, first_shown)])
    reward = (clicked == first_shown).astype(float)            # click on the slot-1 product
    prop1 = df.propensity.str[0].to_numpy().astype(float)
    prop1 = np.where(np.isfinite(prop1) & (prop1 > 0), prop1, np.nan)
    w = np.nan_to_num(np.where(match == 1, np.minimum(1.0 / np.maximum(prop1, 0.02), 10.0), 0.0), nan=0.0)
    return {"hit_rate@10": hit, "IPS_ctr_slot1": (w * reward).mean(),
            "SNIPS_ctr_slot1": (w * reward).sum() / max(w.sum(), 1e-9), "slot1_matches": int(match.sum())}

old_slot1 = (test.clicked == test.shown.str[0]).mean()
print(f"old system, observed slot-1 CTR: {old_slot1:.4f}")
pd.DataFrame({name: evaluate(p, test) for name, p in POLICIES.items()}).T

old system, observed slot-1 CTR: 0.0105


,hit_rate@10,IPS_ctr_slot1,SNIPS_ctr_slot1,slot1_matches
baseline,0.387025,0.006105,0.043850,935.0
coview,0.251678,0.007972,0.026525,1373.0
ours,0.504474,0.011071,0.047306,1369.0


In [7]:
rng = np.random.default_rng(0)
B = 2000
N = len(test)

def per_row(policy, df):
    """Resultados por fila de una política (una sola pasada por el test)."""
    recs = [policy(r) for r in df.itertuples()]
    shown1 = df.shown.str[0].to_numpy()
    clicked = df.clicked.to_numpy()
    prop1 = df.propensity.str[0].to_numpy().astype(float)
    match = np.array([r[0] == s for r, s in zip(recs, shown1)], dtype=float)
    reward = np.array([c == s for c, s in zip(clicked, shown1)], dtype=float)
    prop1 = np.where(np.isfinite(prop1) & (prop1 > 0), prop1, np.nan)
    w = np.where(match == 1, np.minimum(1.0 / np.maximum(prop1, 0.02), 10.0), 0.0)
    w = np.nan_to_num(w, nan=0.0)
    hit = np.array([np.nan if pd.isna(c) else float(c in r)
                    for c, r in zip(clicked, recs)])
    return {"num": w * reward, "w": w, "hit": hit}

R = {name: per_row(p, test) for name, p in POLICIES.items()}

def stats(res, idx):
    """IPS, SNIPS y hit@10 sobre un conjunto de filas (índices)."""
    num, w, hit = res["num"][idx], res["w"][idx], res["hit"][idx]
    snips = num.sum() / w.sum() if w.sum() > 0 else np.nan
    return np.array([num.mean(), snips, np.nanmean(hit)])

# Los mismos índices bootstrap para todas las políticas: así las diferencias son pareadas
IDX = [rng.integers(0, N, N) for _ in range(B)]
ALL = np.arange(N)
names = ["IPS slot1", "SNIPS slot1", "hit@10"]

def summary(name):
    point = stats(R[name], ALL)
    boot = np.array([stats(R[name], i) for i in IDX])
    lo, hi = np.percentile(boot, [2.5, 97.5], axis=0)
    return pd.DataFrame({"valor": point, "IC95_inf": lo, "IC95_sup": hi}, index=names)

for name in POLICIES:
    print(f"\n== {name} ==")
    print(summary(name).round(4))

def diff(a, b):
    """Diferencia pareada a − b con IC95 (si el IC no incluye 0, la diferencia es significativa)."""
    point = stats(R[a], ALL) - stats(R[b], ALL)
    boot = np.array([stats(R[a], i) - stats(R[b], i) for i in IDX])
    lo, hi = np.percentile(boot, [2.5, 97.5], axis=0)
    return pd.DataFrame({"diff": point, "IC95_inf": lo, "IC95_sup": hi}, index=names)

print("\n== ours − baseline ==")
print(diff("ours", "baseline").round(4))


== baseline ==
              valor  IC95_inf  IC95_sup
IPS slot1    0.0061    0.0043    0.0080
SNIPS slot1  0.0439    0.0314    0.0578
hit@10       0.3870    0.3561    0.4181

== coview ==
              valor  IC95_inf  IC95_sup
IPS slot1    0.0080    0.0055    0.0111
SNIPS slot1  0.0265    0.0186    0.0368
hit@10       0.2517    0.2239    0.2818

== ours ==
              valor  IC95_inf  IC95_sup
IPS slot1    0.0111    0.0082    0.0143
SNIPS slot1  0.0473    0.0352    0.0608
hit@10       0.5045    0.4728    0.5385

== ours − baseline ==
               diff  IC95_inf  IC95_sup
IPS slot1    0.0050    0.0026    0.0078
SNIPS slot1  0.0035   -0.0089    0.0174
hit@10       0.1174    0.0874    0.1464


In [8]:
from features import viewed_from_requests

cold_user = test[test.recent_views.apply(lambda v: len(viewed_from_requests(v) or []) == 0)]
print("filas usuario frío:", len(cold_user))

if len(cold_user):
    pd.DataFrame({name: evaluate(p, cold_user) for name, p in POLICIES.items()}).T

filas usuario frío: 5040


In [ ]:
test["purchased"] = test.purchased.replace("", None)

def purchase_hit(policy, df):
    d = df[df.purchased.notna()]
    return float(np.mean([p in policy(r) for p, r in zip(d.purchased, d.itertuples())]))

cold = test[test.recent_views.apply(lambda v: len(viewed_from_requests(v)) == 0)]
print(f"usuarios fríos (sin vistas): {len(cold)} de {len(test)}")
out = {}
for name, p in POLICIES.items():
    out[name] = {"hit@10 compras (todos)": purchase_hit(p, test),
                 "hit@10 clics (todos)": evaluate(p, test)["hit_rate@10"],
                 "hit@10 clics (fríos)": evaluate(p, cold)["hit_rate@10"],
                 "hit@10 compras (fríos)": purchase_hit(p, cold)}
pd.DataFrame(out).T.round(4)

usuarios fríos (sin vistas): 5040 de 25000


,hit@10 compras (todos),hit@10 clics (todos),hit@10 clics (fríos),hit@10 compras (fríos)
baseline,0.5135,0.3870,0.3158,0.5455
coview,0.3063,0.2517,0.3158,0.5455
ours,0.6727,0.5045,0.3421,0.5455


: 

## Next steps

- Few `slot1_matches` means a noisy IPS estimate: report it next to the estimate.
- Personal taste: who bought what in `shop.history_events` (matrix factorisation, item-to-item similarity).
- Cold start: new products only have a `description`. Text embeddings can place them next to similar products.
- Context: does price sensitivity change with `device` and `region`?
- Live: the stream changes (trends, price drops, stock-outs). Your `team_NN.feedback` tells you how you are really doing.